In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sample = pd.read_csv(Path().resolve() / "random_patients_with_labels.csv")
sample

In [ ]:
answers = pd.read_csv(Path().resolve() / "clinical_analysis_answers_2025-03-28 15:06:33.csv", sep=";")
answers

In [ ]:
is_real = [
    sample[sample["REF"] == i].reset_index(drop=True).iloc[0]["is_real"]
    for i in range(1, 25)
]

In [ ]:
answers["is_real"] = is_real

In [ ]:
answers

In [ ]:
answers["Q1"] = answers["Q1"].map({"real": True, "synthetic": False})

In [ ]:
answers

In [ ]:
# Global

correct = 0

for _, row in answers.iterrows():
    correct += row["Q1"] == row["is_real"]

print(f"Total correct: {correct} / {len(answers)}")
print(f"Accuracy: {correct / len(answers)}")
print(f"Mean confidence: {answers['Q2'].mean()}")

In [ ]:
# Real

correct_real = 0

for _, row in answers[answers["is_real"]].iterrows():
    correct_real += row["Q1"] == row["is_real"]

print(f"Real correct: {correct_real} / {len(answers[answers['is_real']])}")
print(f"Accuracy: {correct_real / len(answers[answers['is_real']])}")
print(f"Mean confidence real: {answers[answers['is_real']]['Q2'].mean()}")
real_correct_df = answers[answers['is_real']]
real_correct_df = real_correct_df[real_correct_df['Q1'] == real_correct_df['is_real']]
real_incorrect_df = answers[answers['is_real']]
real_incorrect_df = real_incorrect_df[real_incorrect_df['Q1'] != real_incorrect_df['is_real']]
print(
    f"Mean confidence real correct: {real_correct_df['Q2'].mean()}"
)
print(
    f"Mean confidence real incorrect: {real_incorrect_df['Q2'].mean()}"
)

In [ ]:
# False

correct_false = 0

for _, row in answers[~answers["is_real"]].iterrows():
    correct_false += row["Q1"] == row["is_real"]

print(f"False correct: {correct_false} / {len(answers[~answers['is_real']])}")
print(f"Accuracy: {correct_false / len(answers[~answers['is_real']])}")
print(f"Mean confidence false: {answers[~answers['is_real']]['Q2'].mean()}")
false_correct_df = answers[~answers['is_real']]
false_correct_df = false_correct_df[false_correct_df['Q1'] == false_correct_df['is_real']]
false_incorrect_df = answers[~answers['is_real']]
false_incorrect_df = false_incorrect_df[false_incorrect_df['Q1'] != false_incorrect_df['is_real']]
print(
    f"Mean confidence false correct: {false_correct_df['Q2'].mean()}"
)
print(
    f"Mean confidence false incorrect: {false_incorrect_df['Q2'].mean()}"
)

In [ ]:
confusion_matrix(answers["is_real"], answers["Q1"])

In [ ]:
fig, axs = plt.subplot_mosaic([["Q1", "Q2"]], figsize=(16.3, 7.5))

In [ ]:
cf = confusion_matrix(answers["is_real"], answers["Q1"]) 
sns.heatmap(
    cf,
    annot=True,
    fmt="d",
    cmap="Blues",
    yticklabels=["Synthetic", "Real"],
    xticklabels=["Synthetic", "Real"],
    ax=axs["Q1"],
)

In [ ]:
stats = "\n\nAccuracy={:0.3f}\nPrecision={:0.3f}\nRecall={:0.3f}".format(np.trace(cf) / float(np.sum(cf)), cf[1, 1] / sum(cf[:, 1]), cf[1, 1] / sum(cf[1, :]))

In [ ]:
axs["Q1"].set_xlabel("Predicted Patient Origin" + stats)
axs["Q1"].set_ylabel("Actual Patient Origin")
# axs["Q1"].set_title("Discriminative Task Confusion Matrix")

In [ ]:
answers[~answers["is_real"]][answers["Q1"] == answers["is_real"]][
    "Q2"
].mean()

In [ ]:
mean_confidence = np.array(
    [
        [
            false_correct_df[
                "Q2"
            ].mean(),
            false_incorrect_df[
                "Q2"
            ].mean(),
        ],
        [
            real_incorrect_df[
                "Q2"
            ].mean(),
            real_correct_df[
                "Q2"
            ].mean(),
        ],
    ]
)

In [ ]:
g = sns.heatmap(
    mean_confidence,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    yticklabels=["Synthetic", "Real"],
    xticklabels=["Synthetic", "Real"],
    ax=axs["Q2"],
)

In [ ]:
axs["Q2"].set_xlabel("Predicted Patient Origin")
axs["Q2"].set_ylabel("Actual Patient Origin")
# axs["Q2"].set_title("Mean Confidence Confusion Matrix")

In [ ]:
fig.savefig("clinical_eval_confusion.pdf", bbox_inches="tight")

In [ ]:
answers[(answers["is_real"] == False) & (answers["Q1"] == False)]

In [ ]:
# lets join both
df = answers.drop(columns=["is_real"]).join(sample.set_index("REF"), on="REF")
df

In [ ]:
# only keep rows where Q3 is not nan
df = df[~df["Q3"].isna()]

In [ ]:
df.to_csv("clinical_analysis_joined.csv", index=False, sep=";")